# Phase 4 — Model B training
This notebook clones only the immutable Phase 4 source revision; it never runs `main` or loads ViLexNorm Test.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO = Path('/kaggle/working/VisolexNorm')
PHASE4_BRANCH = 'phase-4/model-b-training'
PHASE4_COMMIT = '4cfc8afda5f9c7e3a853a5fde98e25a5d2341f67'
REPOSITORY_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--branch', PHASE4_BRANCH, '--single-branch', REPOSITORY_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', PHASE4_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert head == PHASE4_COMMIT, f'Expected {PHASE4_COMMIT}, got {head}'
print(f'Running Phase 4 source at {head}')

In [ ]:
%cd {REPO}
!pip install -q -r requirements-kaggle.txt

import platform, torch, transformers
print(platform.python_version(), torch.__version__, transformers.__version__)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'

## Paths
Update these three input paths to match the attached Kaggle datasets.

In [ ]:
DATA = Path('/kaggle/input/visolexnorm-phase3')
MODEL_A = Path('/kaggle/input/visolexnorm-model-a/model_a')
WORK = Path('/kaggle/working')

In [ ]:
!python -m pytest tests/unit/test_model_b_mixture.py tests/contract/test_model_b_contracts.py -q
!python scripts/build_model_b_mixture.py --repo-root {DATA} --config {REPO}/configs/model_b_config.json --phase3-manifest {DATA}/outputs/phase3_manifest.json --output {WORK}/training_mixture_manifest.json

## Smoke gate: exactly 200 gold + 200 pseudo

In [ ]:
!python scripts/train_model_b.py --model-a-checkpoint {MODEL_A} --data-dir {DATA}/data/processed --mixture-manifest {WORK}/training_mixture_manifest.json --work-dir {WORK}/smoke --smoke-test
import json
smoke = json.load(open(WORK/'smoke/outputs/model_b/smoke_test.json'))
assert smoke['passed'] and smoke['composition'] == {'gold': 200, 'pseudo': 200}
smoke

## Full three-epoch run

In [ ]:
!python scripts/train_model_b.py --model-a-checkpoint {MODEL_A} --data-dir {DATA}/data/processed --mixture-manifest {WORK}/training_mixture_manifest.json --work-dir {WORK}

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/model_b_artifacts', 'zip', '/kaggle/working', 'outputs/model_b')
shutil.make_archive('/kaggle/working/model_b_checkpoint', 'zip', '/kaggle/working', 'checkpoints/model_b')
print('Download both zip files from Kaggle Output.')